## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
import sys
def main():
    nums = list(map(int, sys.stdin.buffer.read().split()))
    if not nums:
        return
    n = nums[0]
    fixed_a = nums[1]
    fixed_b = nums[2]
    arr = nums[3:3 + n]
    operations = []
    #定义函数，取二进制最低位的1
    def lowbit(x):
        return x & (-x)

    l = (fixed_a - fixed_b + n) % n
    l = lowbit(l)
    if l == 0:
        l = n

    def magic_swap():

        operations.append(0)

        for i in range(n):
            if arr[i] == fixed_a:
                arr[i] = fixed_b
            elif arr[i] == fixed_b:
                arr[i] = fixed_a

    def add_operation(x):

        x %= n
        if x == 0:
            return
        operations.append(x)

        for i in range(n):
            arr[i] = (arr[i] + x) % n

    def xor_operation(x):

        if x == 0:
            return
        operations.append(-x)
        for i in range(n):
            arr[i] ^= x

    def calculate_pair(x, y):

        delta = (y - x + n - l + n) % n
        value_a = 0
        value_b = 0
        step = n // 2

        while step >= 2 * l:
            if delta >= step:
                delta -= step
                value_b += step // 2
            else:
                value_a += step // 2
            step //= 2

        value_a += n // 2
        value_a += x & (l - 1)
        value_b += x & (l - 1)
        return value_a, value_b

    def swap_position(c, d):

        # 如果在同一组
        if (c // l) % 2 == (d // l) % 2:

            if (c // l) % 2 == 0:
                middle = (c & (l - 1)) + l
            else:
                middle = c & (l - 1)

            swap_position(c, middle)
            swap_position(d, middle)
            swap_position(c, middle)

        else:

            fixed_pa, fixed_pb = calculate_pair(fixed_a, fixed_b)
            current_pc, current_pd = calculate_pair(c, d)
            add_operation((current_pc - c + n) % n)
            xor_operation(current_pc ^ fixed_pa)
            add_operation((fixed_a - fixed_pa + n) % n)
            magic_swap()
            add_operation((fixed_pa - fixed_a + n) % n)
            xor_operation(current_pc ^ fixed_pa)
            add_operation((c - current_pc + n) % n)


    def permutation_check(sub_arr, length):
        if length == 1:
            return True, []
        half = length // 2
        left_part = [sub_arr[i * 2] // 2 for i in range(half)]  # 偶数位置
        right_part = [sub_arr[i * 2 + 1] // 2 for i in range(half)]  # 奇数位置
        left_ok, left_ops = permutation_check(left_part, half)   #递归检查

        if not left_ok:
            return False, []
        right_ok, right_ops = permutation_check(right_part, half)
        if not right_ok:
            return False, []
        result_ops = []

        if sub_arr[0] % 2:

            if length == 2:
                result_ops.append(1)
            else:
                result_ops.append(-1)

        left_xor = 0
        for op in left_ops:
            if op > 0:
                result_ops.append(-1)
                result_ops.append(1)
            else:
                result_ops.append(op * 2)
                left_xor ^= (-op * 2)
        if left_xor:
            result_ops.append(-left_xor)
        right_xor = 0
        for op in right_ops:
            if op > 0:
                result_ops.append(1)
                result_ops.append(-1)
            else:
                result_ops.append(op * 2)
                right_xor ^= (-op * 2)

        # 判断是否合法
        if (right_xor & half) != (left_xor & half):
            return False, []
        if left_xor >= half:
            left_xor -= half
        if right_xor >= half:
            right_xor -= half
        if left_xor != right_xor:
            return False, []
        # 合并连续 xor
        merged = []
        for op in result_ops:
            if not merged:
                merged.append(op)
            else:
                if op < 0 and merged[-1] < 0:
                    merged[-1] = -((-merged[-1]) ^ (-op))
                    if merged[-1] == 0:
                        merged.pop()
                else:
                    merged.append(op)

        return True, merged
    
    if l > 1:
        low_bits = [arr[i] & (l - 1) for i in range(l)]
        ok, operation_list = permutation_check(low_bits, l)
        if not ok:
            print(-1)
            return
        for op in operation_list:
            if op > 0:
                add_operation(op)
            else:
                xor_operation(-op)
    for start in range(l):
        group = []
        for j in range(start, n, l):
            group.append(arr[j])
        group.sort()
        index = 0
        success = True
        
        for j in range(start, n, l):
            if group[index] != j:
                success = False
                break
            index += 1
        if not success:
            print(-1)
            return
        # 调整位置
        for j in range(start, n, l):
            if arr[j] != j:
                swap_position(j, arr[j])
                
    for i in range(n):

        if arr[i] != i:
            print(-1)
            return
    output = [str(len(operations))]
    for op in operations:
        if op == 0:
            output.append("0")
        elif op < 0:
            output.append(f"1 {-op}")
        else:
            output.append(f"2 {op}")
    sys.stdout.write("\n".join(output))
if __name__ == "__main__":
    main()

## B 长跑

In [ ]:
## add your code here
import sys
nums = list(map(int, sys.stdin.read().split()))
k = 0
out = []

while k < len(nums):
    N, L, Maxn, S = nums[k:k+4]    #第一行输入的四个数字N,L,Maxn,S
    k += 4
    station = {}
    for _ in range(N):
        Pi, Ci = nums[k], nums[k+1]
        k += 2
        if Pi not in station or Ci < station[Pi]:
            station[Pi] = Ci

    if Maxn >= L:      #体力上限大于总路径，无需补给可直达终点
        out.append("Yes")
        continue

    place = [0]
    price = [0]
    for p in sorted(station):
        if p < L:
            place.append(p)
            price.append(station[p])
    place.append(L)
    price.append(0)

    m = len(place)
    spend = [10**9] * m
    spend[0] = 0

    ok = False
    for i in range(m):
        if spend[i] > S:   #没有足够的硬币进行补给，到不了终点
            continue
        j = i + 1
        while j < m and place[j] - place[i] <= Maxn: #两补给点之间的距离小于体力
            need = spend[i] + price[j]
            if place[j] == L and spend[i] <= S:
                ok = True
                break
            if need < spend[j]:
                spend[j] = need
            j += 1
        if ok:
            break

    out.append("Yes" if ok else "No")

print("\n".join(out))

## C 最长回文

In [ ]:
## add your code here
import sys

def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return      
    n = int(input_data[0])
    A = input_data[1]
    B = input_data[2]
    if n == 0:
        print(0)
        return

    #字符串哈希参数配置,使用较大的素数规避哈希碰撞
    mod = (1 << 61) - 1
    base = 131
    power = [1] * (n + 1)
    for i in range(1, n + 1):
        power[i] = (power[i - 1] * base) % mod
    def build_hash(s):
        h = [0] * (n + 1)
        for i in range(n):
            h[i + 1] = (h[i] * base + ord(s[i])) % mod
        return h
    A_reverse = A[::-1]
    hash_A_reverse = build_hash(A_reverse)
    hash_B = build_hash(B)

    #哈希匹配函数(o(1))
    def check_match(idx_A, idx_B, L):
        if L == 0: return True
        h1 = (hash_A_reverse[idx_A + L - 1] - hash_A_reverse[idx_A - 1] * power[L]) % mod
        h2 = (hash_B[idx_B + L - 1] - hash_B[idx_B - 1] * power[L]) % mod
        return h1 == h2   
    # Manacher算法，返回数组P(P[i]表示以i为中心的回文半径）
    def manacher(s):
        T = ['#'] * (2 * n + 1)
        for idx in range(n):
            T[2 * idx + 1] = s[idx]
        P = [0] * (2 * n + 1)
        C = 0
        R = 0
        for i in range(2 * n + 1):
            i_mirror = 2 * C - i
            if R > i:
                P[i] = min(R - i, P[i_mirror])
            else:
                P[i] = 0
            while (i - 1 - P[i] >= 0 and 
                   i + 1 + P[i] < 2 * n + 1 and 
                   T[i - 1 - P[i]] == T[i + 1 + P[i]]):
                P[i] += 1
            if i + P[i] > R:
                C = i
                R = i + P[i]
        return P
    P_A = manacher(A)
    P_B = manacher(B)
    max_len = 0

    #回文中心落在A内部
    for j in range(2 * n + 1):
        length = P_A[j]
        if length > max_len:
            max_len = length           
        p = (j - length) // 2 + 1
        i = (j + length) // 2       
        if 1 <= i <= n and p > 1:
            idx_A = n - p + 2
            idx_B = i
            max_ext = min(n - idx_A + 1, n - idx_B + 1)            
            if max_ext <= 0:
                continue                
            #最低门槛剪枝
            req_lcp = (max_len - length) // 2 + 1
            if req_lcp > max_ext:
                continue                
            #只有通过了这 O(1) 的最低门槛测试，我们才值得去二分挖掘它的潜力
            if check_match(idx_A, idx_B, req_lcp):
                low = req_lcp
                high = max_ext
                best_lcp = req_lcp
                while low <= high:
                    mid = (low + high) >> 1
                    if check_match(idx_A, idx_B, mid):
                        best_lcp = mid
                        low = mid + 1
                    else:
                        high = mid - 1
                if length + 2 * best_lcp > max_len:
                    max_len = length + 2 * best_lcp

    #回文中心落在B内部
    for j in range(2 * n + 1):
        length = P_B[j]
        if length > max_len:
            max_len = length            
        i = (j - length) // 2 + 1
        q = (j + length) // 2       
        if 1 <= i <= n and q < n:
            idx_A = n - i + 1
            idx_B = q + 1
            max_ext = min(n - idx_A + 1, n - idx_B + 1)   
            if max_ext <= 0:
                continue
            req_lcp = (max_len - length) // 2 + 1
            if req_lcp > max_ext:
                continue               
            if check_match(idx_A, idx_B, req_lcp):
                low = req_lcp
                high = max_ext
                best_lcp = req_lcp
                while low <= high:
                    mid = (low + high) >> 1
                    if check_match(idx_A, idx_B, mid):
                        best_lcp = mid
                        low = mid + 1
                    else:
                        high = mid - 1
                if length + 2 * best_lcp > max_len:
                    max_len = length + 2 * best_lcp                  
    print(max_len)
if __name__ == '__main__':
    solve()

## D 优惠券

In [ ]:
## add your code here
import sys
lines = sys.stdin.buffer.read().splitlines()
idx = 0
ans = []
while idx < len(lines):
    if not lines[idx].strip():
        idx += 1
        continue
    m = int(lines[idx])
    idx += 1
    cnt = [0] * 100005
    last = [0] * 100005
    bit = [0] * (m + 2)
    def query(i):
        if i <= 0:
            return 0
        s = 0
        while i > 0:
            s += bit[i]
            i -= i & -i
        return s
    def add(i, v):
        while i <= m:
            bit[i] += v
            i += i & -i
    def kth(k):
        p = 0
        step = 1
        while step << 1 <= m:
            step <<= 1

        while step:
            np = p + step
            if np <= m and bit[np] < k:
                p = np
                k -= bit[np]
            step >>= 1

        return p + 1
    
    qnum = 0
    bad = -1
    for row in range(1, m + 1):
        cur = lines[idx].split()
        idx += 1

        if bad != -1:
            continue
        if len(cur) == 1:
            add(row, 1)
            qnum += 1
            continue
        op = cur[0]
        x = int(cur[1])
        if op == b'I':
            cnt[x] += 1
        else:
            cnt[x] -= 1
        if cnt[x] < 0 or cnt[x] > 1:
            before = query(last[x] - 1)
            if qnum - before == 0:
                bad = row
            else:
                p = kth(before + 1)
                add(p, -1)
                qnum -= 1
                if cnt[x] < 0:
                    cnt[x] = 0
                else:
                    cnt[x] = 1
        last[x] = row
    ans.append(str(bad))
print("\n".join(ans))

## E 任意点

In [ ]:
## add your code here
import sys
data = list(map(int, sys.stdin.read().split()))
if not data:
    exit()
n = data[0]
positions = [(data[i], data[i + 1]) for i in range(1, 2 * n, 2)]
start = list(range(n))
def find(x):
    while start[x] != x:
        start[x] = start[start[x]]
        x = start[x]
    return x
ans = n - 1
for i in range(n):
    for j in range(i + 1, n):
        if positions[i][0] == positions[j][0] or positions[i][1] == positions[j][1]:
            ri, rj = find(i), find(j)

            if ri != rj:
                start[ri] = rj
                ans -= 1
print(ans)

## F 通配符匹配

In [ ]:
## add your code here
import sys
def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    pattern = input_data[0]
    n = int(input_data[1])
    filenames = input_data[2:]

    class ExactMatcher:
        def __init__(self, p):
            self.L = len(p)
            self.parts = []
            curr = 0
            for part in p.split('?'):
                if part:
                    self.parts.append((part, curr))
                curr += len(part) + 1

        def match(self, S, start_idx):
            if start_idx < 0 or start_idx + self.L > len(S):
                return False
            for p, off in self.parts:
                if S[start_idx + off : start_idx + off + len(p)] != p:
                    return False
            return True

    class MidMatcher:
        def __init__(self, p):
            self.L = len(p)
            self.parts = []
            curr = 0
            for part in p.split('?'):
                if part:
                    self.parts.append((part, curr))
                curr += len(part) + 1       
        def match_first(self, S, start, end):
            if start + self.L > end:
                return -1
                
            if not self.parts:
                return start
                
            best_part = None
            best_off = -1
            min_count = float('inf')
            for p, off in self.parts:
                c = S.count(p, start, end)
                if c < min_count:
                    min_count = c
                    best_part = p
                    best_off = off
                    
            search_idx = start + best_off
            limit = end - self.L + best_off + len(best_part)
            
            while True:
                idx = S.find(best_part, search_idx, limit)
                if idx == -1:
                    return -1
                    
                base_pos = idx - best_off
                match_ok = True
                for p, off in self.parts:
                    if off == best_off: 
                        continue
                    if S[base_pos + off : base_pos + off + len(p)] != p:
                        match_ok = False
                        break
                        
                if match_ok:
                    return base_pos
                search_idx = idx + 1

    blocks = pattern.split('*')
    out = []

    if len(blocks) == 1:
        matcher = ExactMatcher(blocks[0])
        for s in filenames:
            if len(s) == matcher.L and matcher.match(s, 0):
                out.append("YES")
            else:
                out.append("NO")
        print('\n'.join(out))
        return
    pref = blocks[0]
    suff = blocks[-1]
    mids = blocks[1:-1]
    min_required_len = sum(len(b) for b in blocks)
    pref_matcher = ExactMatcher(pref) if pref else None
    suff_matcher = ExactMatcher(suff) if suff else None
    mid_matchers = [MidMatcher(b) for b in mids if b]
    
    for s in filenames:
        if len(s) < min_required_len:
            out.append("NO")
            continue
            
        if pref_matcher and not pref_matcher.match(s, 0):
            out.append("NO")
            continue
            
        if suff_matcher and not suff_matcher.match(s, len(s) - len(suff)):
            out.append("NO")
            continue

        curr_start = len(pref)
        end_limit = len(s) - len(suff)
        possible = True
        for matcher in mid_matchers:
            pos = matcher.match_first(s, curr_start, end_limit)
            if pos == -1:
                possible = False
                break
            curr_start = pos + matcher.L
        if possible:
            out.append("YES")
        else:
            out.append("NO")
    print('\n'.join(out))

if __name__ == '__main__':
    solve()

## G 汉诺塔

In [ ]:
## add your code here
import sys

def solve():
    try:
        line1 = sys.stdin.readline()
        if not line1: return
        n = int(line1.strip())
        priorities = sys.stdin.readline().split()
    except EOFError:
        return
    name_to_idx = {'A': 0, 'B': 1, 'C': 2}  # 柱子映射 0:A, 1:B, 2:C   
    f = [[0] * 3 for _ in range(n + 1)]  # f[m][i]表示将m个盘子从柱子i移走的步数
    target = [[0] * 3 for _ in range(n + 1)]  # target[m][i]表示将m个盘子从柱子i移走后的终点柱子

    for i in range(3):
        for op in priorities:
            start = name_to_idx[op[0]]
            end = name_to_idx[op[1]]
            if start == i:
                f[1][i] = 1
                target[1][i] = end
                break

    for m in range(2, n + 1):
        for i in range(3):
            t1 = target[m-1][i]
            k = 3 - i - t1  # 找到剩下那根柱子 k
            

            if target[m-1][t1] != i:
                # 刚好移到了大盘子所在的 k 柱子
                f[m][i] = f[m-1][i] + 1 + f[m-1][t1]
                target[m][i] = target[m-1][t1]
            else:
                # 绕回来了，大盘子还得再移一次
                f[m][i] = f[m-1][i] + 1 + f[m-1][t1] + 1 + f[m-1][i]
                target[m][i] = t1

    print(f[n][0])  #从柱子A出发

if __name__ == "__main__":
    solve()

## H 马步距离

In [ ]:
## add your code here
import sys
def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return  
    xp, yp, xs, ys = map(int, input_data)
    
    x, y = abs(xs - xp), abs(ys - yp)  #转换为相对坐标，并取绝对值
    
    if x < y:
        x, y = y, x
    if x == 0 and y == 0:
        print(0)
        return
    if x == 1 and y == 0:
        print(3)
        return
    if x == 1 and y == 1:
        print(2)
        return
    if x == 2 and y == 2:
        print(4)
        return

    res = max((x + 1) // 2, (x + y + 2) // 3)
    
    if (res % 2) != ((x + y) % 2):
        res += 1   
    print(res)
if __name__ == "__main__":
    solve()

## I 直方图最大矩形

In [ ]:
## add your code here
#
# 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
#
# 
# @param heights int整型一维数组 
# @return int整型
#
class Solution:
    def largestRectangleArea(self , heights: List[int]) -> int:
        # write code 
        arr = [0] + heights + [0]
        mono = [0]
        best = 0
        for right in range(1, len(arr)):
            while arr[right] < arr[mono[-1]]:
                mid = mono.pop()
                left = mono[-1]
                area = arr[mid] * (right - left - 1)
                if area > best:
                    best = area
            mono.append(right)
        return best

## J 消防局的设立

In [ ]:
## add your code here
import sys
def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    if n == 0:
        print(0)
        return
    
    parent = [0] * (n + 1)
    for i in range(2, n + 1):
        parent[i] = int(input_data[i - 1])
    
    unc = [0] * (n + 1)   #子树中最远未覆盖节点到i的距离
    st = [float('inf')] * (n + 1)  #子树中最近消防站到i的距离
    
    ans = 0
    # 自底向上遍历
    for i in range(n, 0, -1):
        if unc[i] + st[i] <= 2:  #被覆盖
            unc[i] = -float('inf')
            
        if unc[i] == 2:  #必须建站
            ans += 1
            st[i] = 0
            unc[i] = -float('inf')

        if i > 1:  #向父节点传递信息
            p = parent[i]
            unc[p] = max(unc[p], unc[i] + 1)
            st[p] = min(st[p], st[i] + 1)
    
    if unc[1] >= 0:  #根节点处理
        ans += 1
    print(ans)

if __name__ == '__main__':
    solve()